# **Final Year Project - Year 4 Software Development**
## Iteration 1 - Onset Detection Applied for Chord Transitions

*By Mark Lambert - C00192497*

*19/10/2026*

---

The aim of this notebook is to resolve the findings in ```OnsetDetectionResearch```. The primary issue discovered was that onset_detect returns several onsets per strum.

In order to detect chord transitions, a guitar strum needs to be isolated with a start and end point. But, onset_detect is also quite sensitive, detecting additional noise and classifying the noise as an onset. 


```%pip install matplotlib librosa sounddevice```

In [1]:
import librosa
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
chords, sr = librosa.load('clips/C_to_G.wav', sr=None)

The clip loaded is the C chord transitioning to a G chord

In [3]:
chroma = librosa.feature.chroma_stft(y=chords, sr=sr)

In [4]:
onset_frames = librosa.onset.onset_detect(y=chords, sr=sr)

In [5]:
onset_frames

array([  3,  10,  15,  20,  25,  85, 105, 115, 118, 122, 126, 130, 134,
       218, 222, 241])

The output of ```onset_detect``` shows many onsets, however, the clip has just two guitar strums. 

Because the plectrum strikes more than one strum when playing a chord, onsets that are **close together** can be assumed to be from the same chord strum.

To isolate each strum, first iterate through the onset_frames. The loop starts at one for efficiency, so there is no need for an if to check if there exists a frame ahead of the current one.

Prev holds the start of the chord strum, and cur holds the ending frame for the strum.

Once the frames have been extracted, multidimensional array slicing is used on the ```chroma``` and the average chroma for the strum is calculated. 

```avg_chroma``` is then passed into ```detect_chroma```

In [6]:
from ChordDetect import detect_chord
for i in range (1, len(onset_frames)):
    prev = onset_frames[i-1]
    cur = onset_frames[i]

    # using multi demnsional array slicing, we keep all 12 rows (the pitch class)
    # and slice from frame prev to frame cur
    strum_slice = chroma[:, prev: cur]

    # calculate the average chroma for the slice
    avg_chroma = np.mean(strum_slice, axis=1)

    chord, score = detect_chord(avg_chroma)
    print(chord)
    print(score)

Em
0.5399066723822429
C
0.5659218705194354
C
0.5863850084681571
C
0.6786110779905822
C
0.7049506158770077
Em
0.5182847735801173
G
0.5496453062028683
Em
0.54383754999523
G
0.5481335618301348
G
0.5750298963518834
G
0.6288259600210228
G
0.6764540647043645
G
0.6783145381548951
G
0.5959232431563923
Dm
0.5131142529903873


The extra onsets show open guitar noise at the beginning of the clip and between transitions, where an **Em** code is detected from the open strings ringing. 

There are also many chord detections, due to the onsets created from the plectrum striking subsequent strings

## Issues

- Currently, I iterate through ```onset_frames``` raw and perform chord detection on each slice of onsets. But, because ```onset_detect``` is sensitive, I need to isolate and extract one onset per chord strum
